# Model Development - UCI Bank Marketing Dataset

**Project:** CAP182 Capstone - STADIOEquities First-Deposit Activation  
**Part:** B - Model Development  
**Input:** `data/processed/bank_marketing_features.csv` (output of the feature-engineering stage)

---

This notebook develops and validates a **Gradient Boosting Classifier** using the feature-engineered dataset. The model is trained and evaluated using the same validation framework applied to the Logistic Regression model, allowing the two classification approaches to be compared consistently. No preprocessing or feature engineering is repeated in this notebook; the validated, feature-engineered dataset is loaded and used as provided.

## 1. Load the Feature-Engineered Dataset

The preprocessing and feature-engineering stages have prepared the dataset for model development. The resulting dataset contains numerical predictor variables and the binary target variable `y_binary`.

This notebook develops a Gradient Boosting classification model using the saved feature-engineered dataset.

The UCI Bank Marketing dataset is used as a structural proxy for the STADIOEquities activation problem. The target variable represents whether a client subscribed to a term deposit and is therefore used as a proxy for the project's financial conversion outcome. The model results are not used to make claims about actual STADIOEquities customers.

The feature-engineered dataset is loaded from the processed-data folder so that this notebook can be run independently from the earlier preprocessing and feature-engineering notebooks.

In [1]:
import pandas as pd
import numpy as np

# Load the validated feature-engineered dataset
FEATURES_PATH = "data/processed/bank_marketing_features.csv"

df = pd.read_csv(FEATURES_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully.
Rows: 45211
Columns: 51


**1.1 Verify the Target and Predictor Variables**

The target variable for this classification problem is `y_binary`, where:

- `0` represents the non-conversion outcome
- `1` represents the conversion outcome

All remaining variables are treated as predictor variables.

Before model development, the target distribution and predictor structure are checked to confirm that the feature-engineered dataset has the expected structure.

The same target and predictor definitions used for the Logistic Regression model are retained so that the two models can be compared consistently.

In [2]:
# Separating the target from the predictor variables
X = df.drop(columns=["y_binary"])
y = df["y_binary"]

# Checking the target distribution
print("Target distribution:")
print(y.value_counts())

# Checking the target proportions
print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))

# Confirming the predictor structure
print("\nPredictor dimensions:")
print("Rows:", X.shape[0])
print("Predictors:", X.shape[1])

# Confirming that all predictors are numerical
print("\nNon-numerical predictors:")
print(X.select_dtypes(exclude=["number"]).columns.tolist())

Target distribution:
y_binary
0    39922
1     5289
Name: count, dtype: int64

Target proportions:
y_binary
0    0.883
1    0.117
Name: proportion, dtype: float64

Predictor dimensions:
Rows: 45211
Predictors: 50

Non-numerical predictors:
[]


## 2. Create the Train/Test Split

The dataset is divided into a training set and a held-out test set before model development.

An 80/20 split is used, with 80% of the observations used for model development and 20% reserved for final evaluation.

Stratification is applied using the target variable so that the proportion of conversion and non-conversion observations remains approximately consistent in both datasets.

The same split configuration used for the Logistic Regression model is retained for Gradient Boosting. This allows the two models to be evaluated on the same training and test observations.

The held-out test set is kept separate from model development and will only be used for the final evaluation of the Gradient Boosting model.

In [3]:
from sklearn.model_selection import train_test_split

# Create the same reproducible 80/20 stratified split used for Logistic Regression
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the resulting dataset sizes
print("Training set:")
print("Rows:", X_train.shape[0])

print("\nTest set:")
print("Rows:", X_test.shape[0])

# Check that the target proportions are preserved
print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True).round(4))

Training set:
Rows: 36168

Test set:
Rows: 9043

Training target proportions:
y_binary
0    0.883
1    0.117
Name: proportion, dtype: float64

Test target proportions:
y_binary
0    0.883
1    0.117
Name: proportion, dtype: float64


## 3. Build the Gradient Boosting Model

Gradient Boosting is used as the second classification model because it can model non-linear relationships and interactions between predictor variables.

Unlike Logistic Regression, Gradient Boosting is a tree-based ensemble method and does not require the predictor variables to be standardised before training. The validated numerical features are therefore provided directly to the model.

The same training data and reproducible settings used for Logistic Regression are retained to ensure that the two models can be compared consistently.

The initial model uses the standard Gradient Boosting configuration without additional hyperparameter tuning. Hyperparameter optimisation is not introduced at this stage so that the baseline model can first be evaluated using the same development framework.

In [4]:
from sklearn.ensemble import GradientBoostingClassifier

# Create the Gradient Boosting classifier
gradient_boosting_model = GradientBoostingClassifier(
    random_state=42
)

# Fit the model using the training data only
gradient_boosting_model.fit(X_train, y_train)

print("Gradient Boosting model trained successfully.")

Gradient Boosting model trained successfully.


## 4. Perform 5-Fold Stratified Cross-Validation

Five-fold stratified cross validation is used to assess the consistency of the Gradient Boosting model during model development.

The cross validation is performed on the training data only, while maintaining the proportion of the two target classes within each fold.

The same cross validation structure and evaluation metrics used for Logistic Regression are applied to Gradient Boosting. This allows the models to be compared using a consistent methodology.

The held-out test set remains separate from cross validation and is reserved for the final evaluation of the trained model.

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate

# Creating the same reproducible 5-fold stratified cross-validation scheme
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Use the same evaluation metrics as the Logistic Regression model
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# Evaluating Gradient Boosting across the training folds
cv_results = cross_validate(
    gradient_boosting_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# Displaying the mean cross-validation performance
print("Mean 5-Fold Cross-Validation Results:")

for metric in scoring:
    train_score = cv_results[f"train_{metric}"].mean()
    validation_score = cv_results[f"test_{metric}"].mean()

    print(
        f"{metric.upper()}: "
        f"Training = {train_score:.4f}, "
        f"Validation = {validation_score:.4f}"
    )

Mean 5-Fold Cross-Validation Results:
ACCURACY: Training = 0.8976, Validation = 0.8947
PRECISION: Training = 0.6980, Validation = 0.6588
RECALL: Training = 0.2196, Validation = 0.2075
F1: Training = 0.3341, Validation = 0.3154
ROC_AUC: Training = 0.8046, Validation = 0.7882


The Gradient Boosting model achieved a mean validation accuracy of 89.47%, precision of 65.88%, recall of 20.75%, F1-score of 31.54% and ROC-AUC of 78.82% across the five cross-validation folds.

The training and validation results are relatively close across the reported metrics, although training performance is consistently higher. This provides an initial indication that the model is not showing a large training-validation performance gap during cross-validation.

Compared with the Logistic Regression cross-validation results, Gradient Boosting produced higher validation precision, recall, F1-score and ROC-AUC, as well as slightly higher accuracy. However, these results represent model-development performance only. The final comparison will be based on performance on the same held-out test set.

## 5. Evaluate the Model on the Held-Out Test Set

After cross-validation, the Gradient Boosting model is evaluated using the held-out test set.

The test set was not used during model fitting or cross-validation and therefore provides an independent assessment of how the trained model performs on unseen observations.

The evaluation includes accuracy, precision, recall, F1-score and ROC-AUC. A confusion matrix and classification report are also produced to provide a detailed view of the model's classification performance.

Because the target is imbalanced, particular attention is given to precision, recall and F1-score rather than accuracy alone.

The same evaluation procedure used for Logistic Regression is applied so that the two models can be compared consistently in the Evaluation stage.

In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Generating predictions for the unseen test set
y_test_pred = gradient_boosting_model.predict(X_test)
y_test_prob = gradient_boosting_model.predict_proba(X_test)[:, 1]

# Calculating evaluation metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_prob)

print("Gradient Boosting Test-Set Results:")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")

# Displaying the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

# Displaying the full classification report
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Gradient Boosting Test-Set Results:
Accuracy:  0.8952
Precision: 0.6708
Recall:    0.2042
F1-score:  0.3130
ROC-AUC:   0.7979

Confusion Matrix:
[[7879  106]
 [ 842  216]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.99      0.94      7985
           1       0.67      0.20      0.31      1058

    accuracy                           0.90      9043
   macro avg       0.79      0.60      0.63      9043
weighted avg       0.88      0.90      0.87      9043



**Interpretation of Test-Set Performance**

The Gradient Boosting model achieved an accuracy of 89.52% and a ROC-AUC of 0.7979 on the held-out test set. As with Logistic Regression, accuracy should be interpreted cautiously because the target variable is imbalanced.

The model achieved a precision of 67.08% for class 1, meaning that approximately 67% of the observations predicted as the positive outcome were correctly classified. Recall for class 1 was 20.42%, indicating that the model identified approximately one-fifth of the actual positive cases.

The confusion matrix shows that the model correctly classified 7,879 of the 7,985 class 0 observations and 216 of the 1,058 class 1 observations. Therefore, the model still misses a substantial proportion of positive cases at the default classification threshold.

Compared with the Logistic Regression test results, Gradient Boosting produced slightly higher accuracy, precision, recall, F1-score and ROC-AUC. However, this comparison is descriptive at this stage. The final assessment of the two models will be conducted in the Evaluation section using the same performance criteria and considering the project's objective.

## 6. Interpret Feature Importance

Gradient Boosting does not provide regression coefficients or odds ratios in the same way as Logistic Regression. Instead, the model provides feature importance scores that indicate the relative contribution of each predictor to the model's predictive decisions.

The feature importance values are examined to identify the predictors that contributed most strongly to the model's predictions.

These values describe predictive importance within the fitted model and should not be interpreted as causal effects. A feature with high importance may contain useful predictive information without necessarily causing the observed outcome.

The results are based on the proxy dataset and therefore should not be interpreted as evidence about actual STADIOEquities customer behaviour.

In [9]:
# Extracting feature importance values from the trained Gradient Boosting model
feature_importance = gradient_boosting_model.feature_importances_

# Creating a table linking each predictor to its importance score
importance_table = pd.DataFrame({
    "feature": X_train.columns,
    "importance": feature_importance
})

# Sorting features from highest to lowest importance
importance_table = importance_table.sort_values(
    by="importance",
    ascending=False
)

# Displaying the 10 most important features
print("Top 10 features by importance:")
print(importance_table.head(10).to_string(index=False))

Top 10 features by importance:
         feature  importance
poutcome_success    0.419614
             age    0.092437
 contact_unknown    0.063447
             day    0.060389
       month_mar    0.049677
     housing_yes    0.046790
       month_oct    0.038185
       month_jun    0.036404
         balance    0.027029
       month_sep    0.023940


**Interpretation of Feature Importance**

The Gradient Boosting model identifies `poutcome_success` as the most important predictor, with an importance score of approximately 0.420. This is substantially higher than the next most important feature, `age`, with an importance score of approximately 0.092.

Other relatively important predictors include `contact_unknown`, `day`, `month_mar`, `housing_yes`, `month_oct`, `month_jun`, `balance` and `month_sep`.

The feature importance results indicate which variables contributed most strongly to the predictive decisions made by the Gradient Boosting model. They do not indicate whether a variable is positively or negatively associated with the outcome, nor do they establish that a variable causes the outcome.

The importance of `poutcome_success` suggests that the outcome of a previous campaign contact contains substantial predictive information for the model. This should be interpreted within the context of the UCI Bank Marketing dataset and should not be assumed to represent the same relationship for STADIOEquities customers.

These findings will be considered alongside the Logistic Regression coefficients and the comparative model performance during the Evaluation stage.

## 7. Gradient Boosting Model Development Summary

The Gradient Boosting Classifier was developed using the feature-engineered UCI Bank Marketing dataset as a structural proxy for the STADIOEquities activation problem.

An 80/20 stratified train/test split was used, with 36,168 observations allocated to training and 9,043 observations reserved for final testing. The Gradient Boosting model was fitted using the training data without additional scaling, as tree-based models do not require standardisation of numerical features.

Five-fold stratified cross-validation was performed on the training data. The model achieved a mean validation accuracy of 89.47%, precision of 65.88%, recall of 20.75%, F1-score of 31.54% and ROC-AUC of 78.82%.

On the held-out test set, the model achieved an accuracy of 89.52%, precision of 67.08%, recall of 20.42%, F1-score of 31.30% and ROC-AUC of 79.79%. The similarity between the cross-validation and held-out test results provides an initial indication of consistent performance on unseen data.

Feature importance analysis identified `poutcome_success`, `age`, `contact_unknown`, `day` and several month-related variables among the most important predictors. These measures describe the contribution of variables to the model's predictions and do not establish causal relationships.

The Gradient Boosting results will be carried forward to the Evaluation stage, where they will be compared with the Logistic Regression model using the same performance criteria. The final assessment will also consider the project's objective of identifying customers associated with non-activation, the class imbalance in the dataset, model interpretability and generalisation to unseen data.

Because the UCI Bank Marketing dataset is being used as a structural proxy, these results should not be interpreted as evidence of actual STADIOEquities customer behaviour. They demonstrate the modelling process and provide a basis for evaluating the approaches within the project.

In [10]:
gradient_boosting_summary = pd.DataFrame({
    "metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "cross_validation": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
        cv_results["test_roc_auc"].mean()
    ],
    "test_set": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1,
        test_roc_auc
    ]
})

gradient_boosting_summary

,metric,cross_validation,test_set
0,Accuracy,0.894686,0.895168
1,Precision,0.658757,0.670807
2,Recall,0.207515,0.204159
3,F1-score,0.315355,0.313043
4,ROC-AUC,0.788209,0.797922
